In [16]:
from pathlib import Path
from typing import Literal, Annotated
from typing_extensions import TypedDict
import json
from datetime import datetime
from collections import defaultdict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import MessagesState, StateGraph, END
from langgraph.graph.state import CompiledStateGraph
from langgraph.prebuilt import ToolNode, tools_condition

from langchain_core.tools import tool, render_text_description_and_args
from langchain_core.prompts import (
    ChatPromptTemplate,
)
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.documents import Document

from langchain_openai import ChatOpenAI

from IPython.display import Image, display

In [17]:
from src.core.settings import get_settings

# Clear cached settings so .env values are always re-read
get_settings.cache_clear()

def get_llm() -> ChatOpenAI:
    settings = get_settings()
    return ChatOpenAI(
        model=settings.llm_model,
        api_key=settings.openai_api_key,
        timeout=60,
        max_retries=2,
    )

llm = get_llm()
print(f"LLM: {llm.model_name}, api_key set: {bool(llm.openai_api_key)}")

LLM: gpt-5.4-mini-2026-03-17, api_key set: True


In [18]:
def display_graph(graph: CompiledStateGraph):
    display(Image(graph.get_graph().draw_mermaid_png()))

In [19]:
from notebooks.evaluation.schemas import RetrievedChunk, RetrievedContext, Report as EvalReport
from src.rag.schemas import WeekReport, MonthReport, YearReport

class DateDict(TypedDict, total=False):
    month: int | None
    year: int
    week: int | None

class ReportState(MessagesState, total=False):
    period: Literal['week', 'month', 'year']
    date: DateDict
    user_id: int
    current_entries: list[dict]       # [{goal_id, name, notes}] for current period
    prev_entries: list[dict]          # [{goal_id, name, notes}] for previous period
    sub_period_reports: str | None    # formatted stored weekly/monthly reports (or None)
    avg_productivity: float
    active_days: int
    goal_metrics_block: str
    language: str
    gender: str | None                # "male" | "female" | None
    week_report: WeekReport
    month_report: MonthReport
    year_report: YearReport
    final_report: str                 # model_dump_json() for all three
    tokens_used: int
    generation_time: float
    context: RetrievedContext

## DB

In [20]:
from src.db.models import Goal, User, Entry, Report
from src.db.session import get_async_sessionmaker

from sqlalchemy import select
from sqlalchemy.orm import joinedload

In [21]:
session_factory = get_async_sessionmaker()

# Notes

In [22]:
from datetime import timedelta
from sqlalchemy import extract


In [23]:
from sqlalchemy import extract, func
from datetime import date as date_type
import numpy as np

def prev_date(period: str, date: DateDict) -> DateDict:
    if period == 'week':
        return {**date, 'week': date['week'] - 1}
    elif period == 'month':
        month = date['month']
        year = date['year']
        return {'month': 12 if month == 1 else month - 1,
                'year': year - 1 if month == 1 else year}
    else:
        return {**date, 'year': date['year'] - 1}


def build_where_clause(period: str, date: DateDict, user_id: int):
    user_filter = Goal.user_id == user_id
    if period == 'year':
        return user_filter & (extract('year', Entry.date_note) == date['year'])
    elif period == 'month':
        return (
            user_filter &
            (extract('month', Entry.date_note) == date['month']) &
            (extract('year', Entry.date_note) == date['year'])
        )
    else:
        return (
            user_filter &
            (extract('week', Entry.date_note) == date['week']) &
            (extract('year', Entry.date_note) == date['year'])
        )


async def query_entries(period: str, date: DateDict, user_id: int) -> dict:
    where_clause = build_where_clause(period, date, user_id)

    async with session_factory() as session:
        result = await session.execute(
            select(Goal)
            .options(joinedload(Goal.entries))
            .join(Goal.entries)
            .where(where_clause)
        )
        goals = result.unique().scalars().all()

        metrics = await session.execute(
            select(
                func.round(func.avg(Entry.productivity_score), 2),
                func.count(func.distinct(Entry.date_note)),
            )
            .join(Goal, Entry.goal_id == Goal.id)
            .where(where_clause)
        )
        avg_productivity, active_days = metrics.one()

        raw_entries_result = await session.execute(
            select(Entry)
            .join(Goal, Entry.goal_id == Goal.id)
            .where(where_clause)
            .order_by(Entry.date_note)
        )
        raw_entries = raw_entries_result.scalars().all()

    entries = [
        {'goal_id': goal.id, 'name': goal.title, 'notes': [e.note for e in goal.entries]}
        for goal in goals
    ]

    goal_metrics_lines = []
    for goal in goals:
        if goal.entries:
            active_days_goal = len({e.date_note for e in goal.entries})
            avg_score = sum(e.productivity_score for e in goal.entries) / len(goal.entries)
            goal_metrics_lines.append(
                f'- {goal.id}: {goal.title}: {active_days_goal} active days, avg score {avg_score:.1f}'
            )
    goal_metrics_block = '\n'.join(goal_metrics_lines)

    return {
        'entries': entries,
        'avg_productivity': float(avg_productivity or 0),
        'active_days': int(active_days or 0),
        'goal_metrics_block': goal_metrics_block,
        'raw_entries': raw_entries,
    }


async def query_all_entries_before(period: str, date: DateDict, user_id: int) -> dict:
    """All entries strictly before the start of the current period."""
    if period == 'month':
        before_date = date_type(date['year'], date['month'], 1)
    else:  # year
        before_date = date_type(date['year'], 1, 1)

    where_clause = (Goal.user_id == user_id) & (Entry.date_note < before_date)

    async with session_factory() as session:
        result = await session.execute(
            select(Goal)
            .options(joinedload(Goal.entries))
            .join(Goal.entries)
            .where(where_clause)
        )
        goals = result.unique().scalars().all()

        raw_entries_result = await session.execute(
            select(Entry)
            .join(Goal, Entry.goal_id == Goal.id)
            .where(where_clause)
            .order_by(Entry.date_note)
        )
        raw_entries = raw_entries_result.scalars().all()

    entries = [
        {'goal_id': goal.id, 'name': goal.title, 'notes': [e.note for e in goal.entries]}
        for goal in goals
    ]

    return {'entries': entries, 'raw_entries': raw_entries}


async def query_reports(period: str, date: DateDict, user_id: int) -> list:
    sub_period = 'week' if period == 'month' else 'month'

    async with session_factory() as session:
        stmt = select(Report).where(Report.user_id == user_id, Report.period == sub_period)

        if period == 'month':
            stmt = stmt.where(
                extract('month', Report.period_start) == date['month'],
                extract('year', Report.period_start) == date['year'],
            )
        else:
            stmt = stmt.where(
                extract('year', Report.period_start) == date['year'],
            )

        result = await session.execute(stmt.order_by(Report.period_start))
        return result.scalars().all()


def format_report_entries(reports: list) -> str:
    parts = []
    for r in reports:
        if not r.final_report:
            continue
        data = r.final_report
        period_label = f"{r.period_start.strftime('%d.%m.%Y')} - {r.period_end.strftime('%d.%m.%Y')}"
        summary = data.get('summary', '')
        goals_text = '\n'.join(
            f"  [{g.get('goal_id', '?')}] {g.get('name', '')}: {g.get('summary', '')}"
            for g in data.get('goals', [])
        )
        tone = data.get('tone', {})
        tone_text = f"{tone.get('word', '')} ({tone.get('scale', '')}/7)"
        what_worked = '; '.join(w.get('title', '') for w in data.get('what_worked', []))
        parts.append(
            f'**{period_label}**\n'
            f'Summary: {summary}\n'
            f'Goals:\n{goals_text}\n'
            f'Tone: {tone_text}\n'
            f'What worked: {what_worked}'
        )
    return '\n\n---\n\n'.join(parts) if parts else ''


def make_chunks(raw_entries: list, relevance_score: float = 1.0) -> list[RetrievedChunk]:
    return [
        RetrievedChunk(
            entry_id=e.id,
            text=e.note,
            embedding=list(e.embedding) if e.embedding is not None else [],
            date=e.date_note,
            goal_id=e.goal_id,
            relevance_score=relevance_score,
        )
        for e in raw_entries
    ]


def pool_avg_similarity_from_entries(raw_entries: list) -> float | None:
    """Avg pairwise cosine similarity of the full historical pool."""
    embs = [list(e.embedding) for e in raw_entries if e.embedding is not None]
    if len(embs) < 2:
        return None
    mat = np.array(embs, dtype=np.float32)
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1.0, norms)
    mat = mat / norms
    sim = mat @ mat.T
    n = len(mat)
    mask = np.triu(np.ones((n, n), dtype=bool), k=1)
    return float(sim[mask].mean())


async def retrieve_entries(state: ReportState):
    period = state['period']
    date = state['date']
    user_id = state['user_id']

    current = await query_entries(period, date, user_id)

    async with session_factory() as session:
        user = await session.get(User, user_id)
        language = user.language
        gender = user.gender

    sub_period_reports = None
    if period in ('month', 'year'):
        stored = await query_reports(period, date, user_id)
        formatted = format_report_entries(stored)
        sub_period_reports = formatted if formatted else None

    source_chunks = make_chunks(current['raw_entries'])
    goal_metrics_block = current['goal_metrics_block']

    if period == 'week':
        return {
            'current_entries': current['entries'],
            'prev_entries': [],
            'sub_period_reports': None,
            'avg_productivity': current['avg_productivity'],
            'active_days': current['active_days'],
            'goal_metrics_block': goal_metrics_block,
            'language': language,
            'gender': gender,
            'context': RetrievedContext(
                moment_chunks=None,
                pattern_reports=None,
                source_chunks=source_chunks if source_chunks else None,
                goal_metrics_block=goal_metrics_block or None,
            ),
        }

    # month / year: all entries before current period start
    all_prev = await query_all_entries_before(period, date, user_id)
    moment_chunks = make_chunks(all_prev['raw_entries']) if all_prev['raw_entries'] else None
    total_available_months = (
        len({e.date_note.strftime("%Y-%m") for e in all_prev['raw_entries']})
        if all_prev['raw_entries'] else None
    )
    pool_sim = pool_avg_similarity_from_entries(all_prev['raw_entries'])

    return {
        'current_entries': current['entries'],
        'prev_entries': all_prev['entries'],
        'sub_period_reports': sub_period_reports,
        'avg_productivity': current['avg_productivity'],
        'active_days': current['active_days'],
        'goal_metrics_block': goal_metrics_block,
        'language': language,
        'gender': gender,
        'context': RetrievedContext(
            moment_chunks=moment_chunks,
            pattern_reports=None,
            source_chunks=source_chunks if source_chunks else None,
            total_available_months=total_available_months,
            pool_avg_similarity=pool_sim,
            goal_metrics_block=goal_metrics_block or None,
        ),
    }

## Final generation

In [24]:
from src.rag.prompts.month_report import MONTH_SYSTEM_TEMPLATE, MONTH_USER_TEMPLATE, MONTH_USER_BASELINE
from src.rag.prompts.week_report import WEEK_SYSTEM_TEMPLATE, WEEK_USER_TEMPLATE
from src.rag.prompts.year_report import YEAR_SYSTEM_TEMPLATE, YEAR_USER_TEMPLATE, YEAR_USER_BASELINE
from src.rag.schemas import WeekReport, MonthReport, YearReport

from langchain_core.messages import SystemMessage, HumanMessage

In [25]:
import time
from calendar import month_name as calendar_month_name

def format_entries(entries: list[dict]) -> str:
    parts = []
    for goal in entries:
        notes_text = '\n'.join(f'- {note}' for note in goal['notes'])
        parts.append(f"**{goal['name']}**\n{notes_text}")
    return '\n\n'.join(parts) if parts else 'No entries.'


async def create_report(state: ReportState):
    active_days = state['active_days']
    language = state['language']
    gender = state.get('gender') or 'not specified'
    period = state['period']
    date = state['date']
    goal_metrics_block = state.get('goal_metrics_block', '')
    sub_period_reports = state.get('sub_period_reports')
    prev_text = format_entries(state['prev_entries'])

    if period == 'week':
        goals_block = '\n'.join(f"- {e['name']}" for e in state['current_entries'])
    else:
        goals_block = '\n'.join(f"- {e['goal_id']}: {e['name']}" for e in state['current_entries'])

    if period == 'week':
        week_dt = datetime.fromisocalendar(date['year'], date['week'], 1)
        user_prompt = WEEK_USER_TEMPLATE.format(
            month_name=calendar_month_name[week_dt.month],
            year=date['year'],
            language=language,
            gender=gender,
            goals_block=goals_block,
            goal_metrics_block=goal_metrics_block,
            active_days=active_days,
            period_start=week_dt.strftime('%d.%m.%Y'),
            period_end=(week_dt + timedelta(days=6)).strftime('%d.%m.%Y'),
            current_entries=format_entries(state['current_entries']),
        )
        schema = WeekReport
        state_key = 'week_report'

    elif period == 'month':
        if sub_period_reports:
            user_prompt = MONTH_USER_TEMPLATE.format(
                month_name=calendar_month_name[date['month']],
                year=date['year'],
                language=language,
                gender=gender,
                goals_block=goals_block,
                goal_metrics_block=goal_metrics_block,
                active_days=active_days,
                current_entries=sub_period_reports,
                rag_context=prev_text,
            )
        else:
            user_prompt = MONTH_USER_BASELINE.format(
                month_name=calendar_month_name[date['month']],
                year=date['year'],
                language=language,
                gender=gender,
                goals_block=goals_block,
                goal_metrics_block=goal_metrics_block,
                active_days=active_days,
                current_entries=format_entries(state['current_entries']),
                rag_context=prev_text,
            )
        schema = MonthReport
        state_key = 'month_report'

    else:  # year
        if sub_period_reports:
            user_prompt = YEAR_USER_TEMPLATE.format(
                year=date['year'],
                language=language,
                gender=gender,
                goals_block=goals_block,
                goal_metrics_block=goal_metrics_block,
                active_days=active_days,
                current_entries=sub_period_reports,
                rag_context=prev_text,
            )
        else:
            user_prompt = YEAR_USER_BASELINE.format(
                year=date['year'],
                language=language,
                gender=gender,
                goals_block=goals_block,
                goal_metrics_block=goal_metrics_block,
                active_days=active_days,
                total_entries=sum(len(e['notes']) for e in state['current_entries']),
                current_entries=format_entries(state['current_entries']),
                rag_context=prev_text,
            )
        schema = YearReport
        state_key = 'year_report'

    system_prompt = {
        'week': WEEK_SYSTEM_TEMPLATE,
        'month': MONTH_SYSTEM_TEMPLATE,
        'year': YEAR_SYSTEM_TEMPLATE,
    }[period]

    t_start = time.time()
    result = await llm.with_structured_output(schema, include_raw=True).ainvoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt),
    ])
    generation_time = time.time() - t_start

    report = result['parsed']
    raw_msg = result['raw']
    tokens_used = raw_msg.usage_metadata.get('total_tokens', 0) if raw_msg.usage_metadata else 0

    return {
        state_key: report,
        'final_report': report.model_dump_json(),
        'tokens_used': tokens_used,
        'generation_time': generation_time,
    }

In [26]:
graph = StateGraph(ReportState)
graph.add_node("retrieve_entries", retrieve_entries)
graph.add_node("create_report", create_report)

graph.set_entry_point("retrieve_entries")
graph.add_edge("retrieve_entries", "create_report")
graph.add_edge("create_report", END)

app = graph.compile()


## Testing

In [27]:
from datetime import date as date_type

from notebooks.evaluation.langfuse_config import get_langfuse_client
from notebooks.evaluation.schemas import EvaluationLogger, ReportResult
from notebooks.evaluation.evaluate import evaluate_report


async def get_all_users() -> list:
    async with session_factory() as session:
        result = await session.execute(select(User))
        return result.scalars().all()


async def get_user_goal_text(user_id: int) -> str:
    async with session_factory() as session:
        result = await session.execute(
            select(Goal).where(Goal.user_id == user_id)
        )
        goals = result.scalars().all()
    return ', '.join(g.title for g in goals) if goals else 'personal development'


def state_to_report_result(state: dict, user_id: int) -> ReportResult:
    period = state['period']
    d = state['date']

    if period == 'month':
        period_start = date_type(d['year'], d['month'], 1)
    elif period == 'year':
        period_start = date_type(d['year'], 1, 1)
    else:  # week
        from datetime import datetime
        period_start = datetime.fromisocalendar(d['year'], d['week'], 1).date()

    return ReportResult(
        context=state.get('context'),
        generated_text=state['final_report'],
        tokens_used=state.get('tokens_used', 0),
        final_generation_time=state.get('generation_time', 0.0),
        method='raw_generation',
        user_id=user_id,
        period=period,
        period_start=period_start,
    )

In [28]:
async def get_periods_for_user(user_id: int) -> list[tuple[str, dict]]:
    periods = []
    async with session_factory() as session:
        rows = await session.execute(
            select(extract('week', Entry.date_note).label('week'), extract('year', Entry.date_note).label('year'))
            .join(Goal, Entry.goal_id == Goal.id).where(Goal.user_id == user_id).distinct().order_by('year', 'week')
        )
        for row in rows:
            periods.append(('week', {'week': int(row.week), 'year': int(row.year)}))
        rows = await session.execute(
            select(extract('month', Entry.date_note).label('month'), extract('year', Entry.date_note).label('year'))
            .join(Goal, Entry.goal_id == Goal.id).where(Goal.user_id == user_id).distinct().order_by('year', 'month')
        )
        for row in rows:
            periods.append(('month', {'month': int(row.month), 'year': int(row.year)}))
        rows = await session.execute(
            select(extract('year', Entry.date_note).label('year'))
            .join(Goal, Entry.goal_id == Goal.id).where(Goal.user_id == user_id).distinct().order_by('year')
        )
        for row in rows:
            periods.append(('year', {'year': int(row.year)}))
    return periods


def load_done_sample_ids(path: str) -> set:
    import csv, os
    if not os.path.exists(path):
        return set()
    with open(path, encoding='utf-8') as f:
        return {row['sample_id'] for row in csv.DictReader(f)}


async def run_evaluation():
    import asyncio
    from tqdm.notebook import tqdm
    from datetime import datetime as dt

    EXISTING_CSV = 'results_raw_generation1є.csv'
    OUTPUT_CSV   = 'results_raw_generation.csv'

    done = load_done_sample_ids(EXISTING_CSV)
    print(f'Already done: {len(done)} samples — skipping\n')

    sem = asyncio.Semaphore(3)
    langfuse = get_langfuse_client()
    logger = EvaluationLogger(output_path=OUTPUT_CSV, langfuse=langfuse)

    users = await get_all_users()
    print(f'Users: {[u.id for u in users]}\n')

    def make_sample_id(user_id, period_type, date_dict):
        if period_type == 'month':
            ps = date_type(date_dict['year'], date_dict['month'], 1)
        elif period_type == 'year':
            ps = date_type(date_dict['year'], 1, 1)
        else:
            ps = dt.fromisocalendar(date_dict['year'], date_dict['week'], 1).date()
        return f"{user_id}_{period_type}_{ps.isoformat()}"

    for user in tqdm(users, desc='Users'):
        report_goal = await get_user_goal_text(user.id)
        all_periods = await get_periods_for_user(user.id)

        for target_period in ('week', 'month', 'year'):
            period_subset = [(pt, d) for pt, d in all_periods if pt == target_period]
            if not period_subset:
                continue

            todo = [(pt, d) for pt, d in period_subset
                    if make_sample_id(user.id, pt, d) not in done]
            if not todo:
                print(f'User {user.id} | {target_period}: all done, skipping')
                continue

            print(f'User {user.id} | {target_period}: {len(todo)}/{len(period_subset)} to process...')

            async def run_one(period_type, date_dict):
                async with sem:
                    try:
                        state = await app.ainvoke({
                            'period': period_type,
                            'date': date_dict,
                            'user_id': user.id,
                        })
                        if not state or not state.get('final_report'):
                            print(f'  skip: {period_type} {date_dict}')
                            return None
                        report_result = state_to_report_result(state, user.id)
                        await evaluate_report(
                            report_result=report_result,
                            report_goal=report_goal,
                            logger=logger,
                        )
                        return state
                    except Exception as e:
                        print(f'  err  {period_type} {date_dict}  {e}')
                        return None

            tasks = [asyncio.create_task(run_one(pt, d)) for pt, d in todo]
            pbar = tqdm(total=len(tasks), desc=f'  {target_period}s', leave=False)

            for coro in asyncio.as_completed(tasks):
                state = await coro
                pbar.update(1)
                if state:
                    pbar.set_postfix(tokens=state.get('tokens_used', '?'), time=f"{state.get('generation_time', 0):.1f}s")

            pbar.close()
            logger.save()

    langfuse.flush()
    print(f'\nDone — {len(logger.results)} new results in {OUTPUT_CSV}')
    print(f'Merge with {EXISTING_CSV} to get full dataset ({len(done) + len(logger.results)} total)')

In [29]:
await run_evaluation()

Already done: 354 samples — skipping

2026-04-25 13:21:53,547 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-25 13:21:53,549 INFO sqlalchemy.engine.Engine SELECT "user".id, "user".name, "user".email, "user".language, "user".gender 
FROM "user"
2026-04-25 13:21:53,550 INFO sqlalchemy.engine.Engine [cached since 4774s ago] {}
2026-04-25 13:21:53,584 INFO sqlalchemy.engine.Engine ROLLBACK
Users: [21, 22, 23, 24, 25]



Users:   0%|          | 0/5 [00:00<?, ?it/s]

2026-04-25 13:21:53,593 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-25 13:21:53,594 INFO sqlalchemy.engine.Engine SELECT goal.id, goal.user_id, goal.title, goal.description, goal.deadline, goal.created_at, goal.status 
FROM goal 
WHERE goal.user_id = %(user_id_1)s::INTEGER
2026-04-25 13:21:53,595 INFO sqlalchemy.engine.Engine [cached since 4774s ago] {'user_id_1': 21}
2026-04-25 13:21:53,610 INFO sqlalchemy.engine.Engine ROLLBACK
2026-04-25 13:21:53,614 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-25 13:21:53,616 INFO sqlalchemy.engine.Engine SELECT DISTINCT EXTRACT(week FROM entry.date_note) AS week, EXTRACT(year FROM entry.date_note) AS year 
FROM entry JOIN goal ON entry.goal_id = goal.id 
WHERE goal.user_id = %(user_id_1)s::INTEGER ORDER BY year, week
2026-04-25 13:21:53,616 INFO sqlalchemy.engine.Engine [cached since 4774s ago] {'user_id_1': 21}
2026-04-25 13:21:53,622 INFO sqlalchemy.engine.Engine SELECT DISTINCT EXTRACT(month FROM entry.date_note) AS month, 

In [30]:
import pandas as pd

df = pd.concat([
    pd.read_csv('results_raw_generation1є.csv'),
    pd.read_csv('results_raw_generation.csv'),
]).drop_duplicates(subset='sample_id').reset_index(drop=True)

df.to_csv('results_raw_generation_full.csv', index=False)
print(f'Total: {len(df)} rows')

FileNotFoundError: [Errno 2] No such file or directory: 'results_raw_generation.csv'